# Komputerowe wspomaganie tłumaczenia

# Zajęcia 11 - urównoleglanie

Na poprzednich zajęciach poznaliśmy techniki pozyskiwania tekstu z Internetu. Jeśli uda nam się w ten sposób pozyskać tekst w jednym języku oraz jego tłumaczenie na inny język, jesteśmy tylko o krok od uzyskania najbardziej przydatnego zasobu z punktu widzenia wspomagania tłumaczenia - pamięci tłumaczeń. Krokiem tym jest automatyczne urównoleglanie tekstu.

Automatyczne urównoleglanie tekstu składa się z dwóch kroków:
1. Podziału tekstu źródłowego oraz docelowego na zdania.
2. Dopasowaniu zdań źródłowych do docelowych.

Zdania, o których mowa w punkcie 1., powinniśmy rozumieć jako segmenty, tj. niekoniecznie kompletne zdania w sensie gramatycznym. Standardowym sposobem podziału tekstu na segmenty jest dzielenie po znaku nowej linii lub zaraz po kropce, o ile jest ona częścią sekwencji: ".[spacja][Wielka litera]"

### Ćwiczenie 1: Zaimplementuj podstawowy algorytm segmentacji tekstu. Użyj odpowiedniego wyrażenia regularnego, łapiącego wielkie litery w dowolnym języku, np. "Ż" (użyj klasy unikodowej). Zwróć listę segmentów.

In [2]:
import re

def sentence_split(text):
    pattern = r'(?<=[.!?])\s+(?=[A-ZÀ-Ž])'
    segments = re.split(pattern, text)
    return [s.strip() for s in segments if s.strip()]


### Ćwiczenie 2: Uruchom powyższy algorytm na treści wybranej przez siebie strony internetowej (do ściągnięcia treści strony wykorzystaj kod z laboratoriów nr 7). Zidentyfikuj co najmniej dwa wyjątki od ogólnej reguły podziału na segmenty i ulepsz algorytm.

In [10]:
import requests
from bs4 import BeautifulSoup

url='https://epoznan.pl'

page = requests.get(url)
soup = BeautifulSoup(page.content, 'html.parser')

headers = soup.find_all('h3', {'class':'postItem__title'})

txt = '\n'.join([header.get_text() for header in headers])
print(txt)

Nie żyje strażak ochotnik. Działał w dwóch OSP pod Poznaniem
To on rozświetlił niebo także nad Poznaniem. Waży 3 kilogramy i tak wygląda!
Zwłoki w Warcie w regionie. Nie można nawet określić płci denata
Syrena alarmowa postawiła na nogi mieszkańców poznańskiego osiedla. Jest komunikat CZK
Pestkownica - rezydencja nad jeziorem. Tam, gdzie spotkania nabierają znaczenia
Gang handlujący ludźmi rozbity w Wielkopolsce. Zatrzymano 9 osób
Wypadek na autostradowej obwodnicy Poznania. Utrudnienia
Latami budowali kościół pod Poznaniem. W weekend konsekrował go arcybiskup
Akcja służb w firmie kurierskiej. Przejęto ponad 400 kg kontrabandy
Wybory i decyzje związane z pogrzebem - Universum radzi
Hołd dla Łukasza Litewki w Poznaniu. Zapłonęły znicze
Rusztowanie przy poznańskim Ratuszu
IMGW znów ostrzega. Alerty pierwszego stopnia dotyczą również Poznania i powiatu
4:0 na stadionie w Poznaniu!
Rekrutacja do przedszkola terapeutycznego w Poznaniu i okolicach
Wypadek w Poznaniu. "Kierujący BMW zjechał n

In [14]:
def sentence_split_enhanced(text):
    abbreviations = r'(prof|dr|ul|str|itp|itd|np|al|mgr)'

    text = re.sub(rf'{abbreviations}\.', r'\1@@@', text, flags=re.IGNORECASE)

    pattern = r'(?<=[.!?])\s+(?=[A-ZÀ-Ž])'
    segments = re.split(pattern, text)

    segments = [s.replace('@@@', '.') for s in segments]

    return [s.strip() for s in segments if s.strip()]

In [15]:
sentence_split(txt)

['Nie żyje strażak ochotnik.',
 'Działał w dwóch OSP pod Poznaniem\nTo on rozświetlił niebo także nad Poznaniem.',
 'Waży 3 kilogramy i tak wygląda!',
 'Zwłoki w Warcie w regionie.',
 'Nie można nawet określić płci denata\nSyrena alarmowa postawiła na nogi mieszkańców poznańskiego osiedla.',
 'Jest komunikat CZK\nPestkownica - rezydencja nad jeziorem.',
 'Tam, gdzie spotkania nabierają znaczenia\nGang handlujący ludźmi rozbity w Wielkopolsce.',
 'Zatrzymano 9 osób\nWypadek na autostradowej obwodnicy Poznania.',
 'Utrudnienia\nLatami budowali kościół pod Poznaniem.',
 'W weekend konsekrował go arcybiskup\nAkcja służb w firmie kurierskiej.',
 'Przejęto ponad 400 kg kontrabandy\nWybory i decyzje związane z pogrzebem - Universum radzi\nHołd dla Łukasza Litewki w Poznaniu.',
 'Zapłonęły znicze\nRusztowanie przy poznańskim Ratuszu\nIMGW znów ostrzega.',
 'Alerty pierwszego stopnia dotyczą również Poznania i powiatu\n4:0 na stadionie w Poznaniu!',
 'Rekrutacja do przedszkola terapeutycznego w

Po podziale tekstu na segmenty po stronie źródłowej oraz docelowej, możemy przystąpić do kroku drugiego - dopasowania segmentów. Głównym wyzwaniem tego kroku jest fakt, iż po stronie źródłowej może być inna liczba segmentów, niż po stronie docelowej. Takie rozbieżności są bardzo częste, a wynikają między innymi z:
* tłumaczenia jednego zdania źródłowego przy użyciu więcej niż jednego zdania
* tłumaczenia więcej niż jednego zdania źródłowego przy użyciu jednego zdania
* pominięcia zdania podczas tłumaczenia
* rozbieżności pomiędzy wersjami tekstu źródłowego i docelowego (np. tekst źródłowy mógł być modyfikowany po przetłumaczeniu i tłumaczenie nie zostało zaktualizowane)
* przetłumaczenia tekstu źródłowego tylko częściowo

Problemy te rozwiązwyane są na różne sposoby. Najpopularniejszym programem do przeprowadzania urównoleglania jest [Hunalign](https://github.com/danielvarga/hunalign). Wejściem do programu są dwa pliki, zawierające po jednym segmencie w linii. Wyjściem - plik urównoleglony w wewnętrznym formacie hunaligna.

### Ćwiczenie 3: Odnajdź dowolną stronę, która jest dostępna w wielu językach. Pobierz z tej strony tekst oryginalny (tylko ze strony głównej) oraz przetłumaczony na dowolny inny język. Przy użyciu Pythona przygotuj pliki dla Hunaligna i uruchom go.

Wyjściem z Hunaligna jest plik w specjalnym formacie Hunaligna. Problem jednak w tym, że niestety nie można go w prosty sposób zaimportować do jakiegokolwiek narzędzia typu CAT. Potrzebna jest konwersja do któregoś z bardziej popularnych formatów, np. XLIFF.

XLIFF jest formatem do przechowywania pamięci tłumaczeń, który opiera się na XML-u. Przykładowy plik XLIFF wygląda następująco:

In [26]:
import requests
from bs4 import BeautifulSoup

def get_segments_from_url(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    response.encoding = 'utf-8'

    soup = BeautifulSoup(response.text, 'html.parser')

    for element in soup(['script', 'style', 'nav', 'header', 'footer', 'aside']):
        element.decompose()

    abbreviations = r'(prof|dr|ul|str|itp|itd|np|al|mgr|ok|pl|en|com)'

    segments = []
    for tag in soup.find_all(['h1', 'h2', 'h3', 'p', 'li']):
        text = tag.get_text(strip=True)
        if not text:
            continue

        protected_text = re.sub(rf'\b{abbreviations}\.', r'\1@@@', text, flags=re.IGNORECASE)

        split_pattern = r'(?<=[.!?])\s+(?=[A-ZÀ-Ž])'
        raw_chunks = re.split(split_pattern, protected_text)

        for chunk in raw_chunks:
            clean_chunk = chunk.replace('@@@', '.').strip()
            if len(clean_chunk) > 1:
                segments.append(clean_chunk)

    return segments

pl_segments = get_segments_from_url("https://www.dhl.com/pl-pl/ecommerce/o-dhl-ecommerce/wazne-informacje.html")
en_segments = get_segments_from_url("https://www.dhl.com/pl-en/ecommerce/about-dhl-ecommerce/news.html")

In [27]:
def save_for_hunalign(segments, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for seg in segments:
            f.write(seg + '\n')


In [28]:
save_for_hunalign(sentence_split_enhanced("\n".join([seg for seg in pl_segments])), "./source.txt")
save_for_hunalign(sentence_split_enhanced("\n".join([seg for seg in en_segments])), "./target.txt")

### Ćwiczenie 4: Napisz konwerter formatu hunaligna na XLIFF.

In [29]:
import xml.etree.ElementTree as ET
from xml.dom import minidom

def convert2xliff(hunalign_file_name, output_xliff, src_lang="en", tgt_lang="pl"):
    xliff = ET.Element("xliff", version="1.2", xmlns="urn:oasis:names:tc:xliff:document:1.2")
    file_tag = ET.SubElement(xliff, "file", {
        "datatype": "plaintext",
        "original": "self",
        "source-language": src_lang,
        "target-language": tgt_lang
    })
    body = ET.SubElement(file_tag, "body")

    with open(hunalign_file_name, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                src_text = parts[0]
                tgt_text = parts[1]
                if src_text == "~~~" or tgt_text == "~~~":
                    continue
                trans_unit = ET.SubElement(body, "trans-unit")
                source = ET.SubElement(trans_unit, "source")
                source.text = src_text
                target = ET.SubElement(trans_unit, "target")
                target.text = tgt_text

    xml_str = ET.tostring(xliff, encoding='utf-8')
    pretty_xml = minidom.parseString(xml_str).toprettyxml(indent="    ")

    with open(output_xliff, "w", encoding="utf-8") as f:
        f.write(pretty_xml)
